In [3]:
# -*- coding: utf-8 -*-
# 【CP1-07 多 Schema 协作】input/state/output + 私有状态的视图隔离
# 文件：CP1/07_multi_schema.ipynb
# 作用：本 Cell 演示 【CP1-07 多 Schema 协作】input/state/output + 私有状态的视图隔离 的完整可运行示例
# 阅读顺序：状态定义 → 节点定义 → 图构建 → 编译执行 → 结果观察

from langgraph.graph import StateGraph,START,END    
from typing import TypedDict, Annotated

# 输入状态
class InputState(TypedDict):
    username: str

# 输出状态
class OutputState(TypedDict):
    graph_output: str

# 全局状态
class OverAllState(TypedDict):
    username: str
    graph_output: str
    nickname: str

# 私有状态
class PrivateState(TypedDict):
    greeting: str

# 第一个节点，对接start ——> InputState 修改的状态内容在全局状态中 -—> OverAllState
def node_1(state : InputState) -> OverAllState:
    # 向全局状态添加username
    return {
        "nickname": "Dear " + state["username"],
    }

# 第二个节点，对接OverAllState 修改的状态内容在私有状态中 -—> PrivateState
def node_2(state : OverAllState) -> PrivateState:
    # 向私有状态添加greeting
    return {
        "greeting": "Hello, " + state["nickname"],
    }

# 第三个节点，对接PrivateState 修改的状态内容在输出状态中 -—> OutputState
def node_3(state : PrivateState) -> OutputState:
    # 向输出状态添加graph_output
    return {
        "graph_output": state["greeting"]+"很高兴认识你！"
    }

# 构建状态图
# 定义图的时候，加载全局状态，输入状态，输出状态
builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)  # 创建状态图构建器：绑定状态 Schema
builder.add_node("node_1", node_1)  # 注册节点到图中
builder.add_node("node_2", node_2)  # 注册节点到图中
builder.add_node("node_3", node_3)  # 注册节点到图中
builder.add_edge(START, "node_1")  # 起点扇出
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)  # 汇入终点

# invoke状态图
graph = builder.compile()  # 编译图：蓝图→可执行对象

result = graph.invoke({"username": "GuiGU"})  # 触发图执行：传入初始 State + config
print(result)


{'graph_output': 'Hello, Dear GuiGU很高兴认识你！'}
